# Feed Forward Network（活性化関数）

このノートブックでは、エンコーダの後半部分である
**Feed Forward Network** を学びます。

書籍 4-12 節（図4.41〜4.49、式4-9, 4-10）の内容をカバーします。

## 目次
1. Feed Forward の位置づけ（図4.42）
2. 基礎知識の復習：行列積と ReLU（図4.41）
3. Feed Forward の式（式4-9）
4. ステップ1: x'W₁ の計算（図4.43）— 6次元 → 24次元に拡張
5. ステップ2: バイアス b₁ の加算（図4.44）
6. ステップ3: ReLU で負の値を0にする（図4.45、式4-10）
7. ステップ4: W₂ を掛けて元の次元に戻す（図4.46）
8. ステップ5: バイアス b₂ の加算（図4.47）
9. 全ステップを一気通貫で実行する
10. Position-wise：トークンごとに独立して適用
11. エンコーダの全体像（図4.48, 4.49）— N回繰り返し
12. まとめ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 日本語フォント設定（macOS）
plt.rcParams['font.family'] = 'Hiragino Sans'
plt.rcParams['axes.unicode_minus'] = False

# 書籍と同じ設定
n_tokens = 7       # トークン数
d_model = 6         # 埋め込み次元
d_ff = 24           # Feed Forward の中間次元（= 4 × d_model）

words = ["Mount", "Fuji", "looks", "beautiful", "in", "spring", "."]

print(f"トークン数: {n_tokens}")
print(f"埋め込み次元 d_model: {d_model}")
print(f"Feed Forward 中間次元 d_ff: {d_ff}  (= 4 × {d_model})")

## 1. Feed Forward の位置づけ（図4.42）

エンコーダの処理の流れを思い出しましょう。

```
入力 X (7×6)
  ↓ Positional Encoding
  ↓
  ┌─────────────────────────────── ×N 回繰り返し ─┐
  │                                              │
  │  Multi-Head Attention                        │
  │       ↓                                      │
  │  Add & Norm  ← 前回のノートブックで学んだ      │
  │       ↓                                      │
  │  ★ Feed Forward ★  ← 今回学ぶ               │
  │       ↓                                      │
  │  Add & Norm                                  │
  │                                              │
  └──────────────────────────────────────────────┘
  ↓
  エンコーダの出力 (7×6)
```

Feed Forward は **Multi-Head Attention + Add & Norm** の後に来る処理です。
その後にもう一度 **Add & Norm** を通ります。

## 2. 基礎知識の復習：行列積と ReLU（図4.41）

Feed Forward で使う数学は2つだけです。

| 手法 | 分類 | 内容 |
|------|------|------|
| **行列の積** | 基礎となる数理的手法 | $(m \times n) \times (n \times p) = (m \times p)$ |
| **ReLU関数** | 情報工学的なアプローチ | 負の値を0に、正の値はそのまま |

### ReLU 関数（式4-10）

$$f(x'_1) = \begin{cases} 0 & (x'_1 < 0) \\ x'_1 & (x'_1 \geq 0) \end{cases}$$

つまり「**負の値は0にして、正の値はそのまま通す**」だけの単純な関数です。
第3章でも登場した活性化関数です。

In [ ]:
# ReLU 関数の可視化

def relu(x):
    """ReLU 関数: max(0, x)"""
    return np.maximum(0, x)

x = np.linspace(-3, 3, 100)
y = relu(x)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, y, linewidth=2.5, color='#e74c3c')
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axvline(x=0, color='gray', linewidth=0.5)
ax.set_xlabel('入力 x', fontsize=12)
ax.set_ylabel('出力 f(x)', fontsize=12)
ax.set_title('式(4-10): ReLU 関数 — f(x) = max(0, x)', fontsize=13, fontweight='bold')
ax.set_xlim(-3, 3)
ax.set_ylim(-1, 3)

# 注釈
ax.annotate('x < 0 → 出力は 0', xy=(-2, 0), xytext=(-2.5, 1.5),
            fontsize=11, arrowprops=dict(arrowstyle='->', color='blue'),
            color='blue', fontweight='bold')
ax.annotate('x ≥ 0 → 出力はそのまま x', xy=(1.5, 1.5), xytext=(0.5, 2.5),
            fontsize=11, arrowprops=dict(arrowstyle='->', color='green'),
            color='green', fontweight='bold')

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 具体例
examples = [2.5, 0.3, 0, -1.2, -0.5, 3.0]
print("具体例:")
for val in examples:
    print(f"  ReLU({val:5.1f}) = max(0, {val:5.1f}) = {relu(val):5.1f}")

## 3. Feed Forward の式（式4-9）

Feed Forward の計算は以下の1つの式で表されます：

$$FFN(\boldsymbol{x'}) = \max(0, \underbrace{\boldsymbol{x'} W_1 + b_1}_{\text{ReLUの入力}}) W_2 + b_2 \quad \text{（式4-9）}$$

### 各記号の意味

| 記号 | 形状 | 意味 |
|------|------|------|
| $\boldsymbol{x'}$ | 1×6 | 入力（7×6 行列の1行分 = 1トークンのベクトル）|
| $W_1$ | 6×24 | 重みパラメータ行列（次元を拡張）|
| $b_1$ | 1×24 | バイアス項 |
| $\max(0, \cdot)$ | — | ReLU 関数（負→0、正→そのまま）|
| $W_2$ | 24×6 | 重みパラメータ行列（次元を戻す）|
| $b_2$ | 1×6 | バイアス項 |

### なぜ 24 次元？

中間次元 24 = **4 × d_model** (4 × 6) です。

一般的なニューラルネットワークでは中間層を小さくすることが多いですが、
**Transformer では逆に4倍に拡大** します。これが Transformer の面白いところです。

多くのパラメータで計算処理することで **精度を高める** ためです。

### Position-wise Feed Forward

この処理は **Position-wise Feed Forward** と呼ばれます。
「Position-wise」= **トークンごとに独立して** 同じ処理を適用するという意味です。

In [ ]:
# 計算の流れを図示

print("=== Feed Forward の計算フロー（1トークン分）===")
print()
print("  x' (1×6)")
print("    │")
print("    ↓  × W₁ (6×24)")
print("  x'W₁ (1×24)         ← 6次元から24次元に拡張")
print("    │")
print("    ↓  + b₁ (1×24)")
print("  x'W₁ + b₁ (1×24)    ← バイアス加算")
print("    │")
print("    ↓  ReLU")
print("  max(0, x'W₁+b₁) (1×24)  ← 負の値を0にする")
print("    │")
print("    ↓  × W₂ (24×6)")
print("  ... × W₂ (1×6)      ← 24次元から6次元に戻す")
print("    │")
print("    ↓  + b₂ (1×6)")
print("  FFN(x') (1×6)        ← 最終出力")
print()
print("ポイント: 6次元 → 24次元 → 6次元（膨らませてから戻す）")

## 4. ステップ1: x'W₁ の計算（図4.43）— 6次元 → 24次元に拡張

まず入力ベクトル x'（1×6）に重み行列 W₁（6×24）を掛けます。

$$\underbrace{\boldsymbol{x'}}_{1 \times 6} \times \underbrace{W_1}_{6 \times 24} = \underbrace{\boldsymbol{x'} W_1}_{1 \times 24}$$

6次元のベクトルが **24次元に拡張** されます。

In [ ]:
# Add & Norm の出力をダミーデータで作成（前回のノートブックの続き）
np.random.seed(42)
X_input = np.round(np.random.randn(n_tokens, d_model) * 0.5, 2)

# Feed Forward のパラメータを初期化
np.random.seed(123)
W1 = np.round(np.random.randn(d_model, d_ff) * 0.3, 3)  # (6×24)
b1 = np.round(np.random.randn(1, d_ff) * 0.1, 3)         # (1×24)
W2 = np.round(np.random.randn(d_ff, d_model) * 0.3, 3)   # (24×6)
b2 = np.round(np.random.randn(1, d_model) * 0.1, 3)       # (1×6)

print("=== パラメータの形状 ===")
print(f"入力 X: {X_input.shape}  (7×6)")
print(f"W₁:     {W1.shape}  (6×24)")
print(f"b₁:     {b1.shape}  (1×24)")
print(f"W₂:     {W2.shape}  (24×6)")
print(f"b₂:     {b2.shape}  (1×6)")
print()

# 1つのトークンで Step 1 を実演（"Mount" = 0番目）
token_idx = 0
x_prime = X_input[token_idx:token_idx+1]  # (1×6) に reshape
print(f'--- "{words[token_idx]}" のステップ1 ---')
print(f"x' = {x_prime[0]}  (1×6)")
print()

# Step 1: x' × W₁
step1 = x_prime @ W1  # (1×6) × (6×24) = (1×24)
print(f"x'W₁ の形状: {step1.shape}  (1×24)  ← 6次元から24次元に拡張！")
print(f"x'W₁ = {np.round(step1[0], 3)}")

## 5. ステップ2: バイアス b₁ の加算（図4.44）

行列積の結果にバイアス項 $b_1$（1×24）を足します。

$$\boldsymbol{x'} W_1 + b_1 \quad (1 \times 24)$$

バイアスは各ニューロンに追加する「底上げ」のようなもので、
学習で最適な値に調整されます。

In [ ]:
# Step 2: バイアス b₁ を加算

step2 = step1 + b1  # (1×24) + (1×24) = (1×24)

print(f'--- "{words[token_idx]}" のステップ2 ---')
print(f"x'W₁      = {np.round(step1[0][:8], 3)} ...（最初の8要素を表示）")
print(f"b₁         = {np.round(b1[0][:8], 3)} ...")
print(f"x'W₁ + b₁  = {np.round(step2[0][:8], 3)} ...")
print(f"形状: {step2.shape}  (1×24)")

## 6. ステップ3: ReLU で負の値を0にする（図4.45、式4-10）

$$\max(0, \boldsymbol{x'} W_1 + b_1)$$

24次元の各要素に対して：
- **0未満の要素 → 0** に変換
- **0以上の要素 → そのまま**

これが活性化関数 ReLU の役割です。

In [ ]:
# Step 3: ReLU を適用

step3 = relu(step2)  # max(0, x'W₁ + b₁)

print(f'--- "{words[token_idx]}" のステップ3（ReLU）---')
print()
print("ReLU 前後の比較（24次元すべて表示）:")
print(f"{'要素':>4s}  {'ReLU前':>8s}  {'ReLU後':>8s}  {'変化':>6s}")
print("-" * 35)
n_zeroed = 0
for i in range(d_ff):
    before = step2[0, i]
    after = step3[0, i]
    change = '→ 0' if before < 0 else 'そのまま'
    if before < 0:
        n_zeroed += 1
    print(f"  [{i:2d}]  {before:8.3f}  {after:8.3f}  {change}")

print(f"\n0になった要素: {n_zeroed}/{d_ff}")
print(f"残った要素: {d_ff - n_zeroed}/{d_ff}")
print(f"形状: {step3.shape}  (1×24)")

In [ ]:
# ReLU の効果を可視化

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# ReLU 前
ax = axes[0]
colors_before = ['#e74c3c' if v < 0 else '#3498db' for v in step2[0]]
ax.bar(range(d_ff), step2[0], color=colors_before, edgecolor='black', linewidth=0.5)
ax.axhline(y=0, color='black', linewidth=1)
ax.set_title('ReLU 前: x\'W₁ + b₁（1×24）', fontsize=12, fontweight='bold')
ax.set_ylabel('値', fontsize=10)
ax.set_xlim(-0.5, d_ff - 0.5)
# 凡例
red_patch = mpatches.Patch(color='#e74c3c', label='負の値（ReLU で 0 になる）')
blue_patch = mpatches.Patch(color='#3498db', label='正の値（そのまま通る）')
ax.legend(handles=[red_patch, blue_patch], fontsize=10)

# ReLU 後
ax = axes[1]
colors_after = ['lightgray' if v == 0 else '#3498db' for v in step3[0]]
ax.bar(range(d_ff), step3[0], color=colors_after, edgecolor='black', linewidth=0.5)
ax.axhline(y=0, color='black', linewidth=1)
ax.set_title('ReLU 後: max(0, x\'W₁ + b₁)（1×24）', fontsize=12, fontweight='bold')
ax.set_xlabel('次元', fontsize=10)
ax.set_ylabel('値', fontsize=10)
ax.set_xlim(-0.5, d_ff - 0.5)
gray_patch = mpatches.Patch(color='lightgray', label='0 になった要素')
blue_patch = mpatches.Patch(color='#3498db', label='残った要素')
ax.legend(handles=[gray_patch, blue_patch], fontsize=10)

plt.suptitle('図4.45: ReLU による処理', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. ステップ4: W₂ を掛けて元の次元に戻す（図4.46）

ReLU の出力（1×24）に W₂（24×6）を掛けて、元の6次元に戻します。

$$\underbrace{\max(0, \boldsymbol{x'} W_1 + b_1)}_{1 \times 24} \times \underbrace{W_2}_{24 \times 6} = \underbrace{\text{出力}}_{1 \times 6}$$

24次元に拡張して処理した情報を、6次元に圧縮して戻すイメージです。

In [ ]:
# Step 4: W₂ を掛ける

step4 = step3 @ W2  # (1×24) × (24×6) = (1×6)

print(f'--- "{words[token_idx]}" のステップ4 ---')
print(f"ReLU出力の形状: {step3.shape}  (1×24)")
print(f"W₂ の形状:      {W2.shape}  (24×6)")
print(f"出力の形状:      {step4.shape}  (1×6)  ← 6次元に戻った！")
print(f"\n結果: {np.round(step4[0], 4)}")

## 8. ステップ5: バイアス b₂ の加算（図4.47）

最後にバイアス $b_2$（1×6）を足して完成です。

$$FFN(\boldsymbol{x'}) = \max(0, \boldsymbol{x'} W_1 + b_1) W_2 + b_2 \quad (1 \times 6)$$

In [ ]:
# Step 5: バイアス b₂ を加算

step5 = step4 + b2  # (1×6) + (1×6) = (1×6)

print(f'--- "{words[token_idx]}" のステップ5（最終出力）---')
print(f"W₂掛け算後: {np.round(step4[0], 4)}")
print(f"b₂:          {np.round(b2[0], 4)}")
print(f"FFN(x'):     {np.round(step5[0], 4)}  ← 最終出力（1×6）")
print()
print(f"入力: {x_prime[0]}  (1×6)")
print(f"出力: {np.round(step5[0], 4)}  (1×6)")
print()
print("形状が同じ 1×6 → 入出力の形状が変わらない！")

## 9. 全ステップを一気通貫で実行する

5つのステップを関数にまとめて、全7トークンに対して実行しましょう。

In [ ]:
# Feed Forward Network の実装

def feed_forward(x, W1, b1, W2, b2):
    """Feed Forward Network（式4-9）
    
    FFN(x') = max(0, x'W₁ + b₁)W₂ + b₂
    
    Parameters:
        x:  入力 (n_tokens × d_model)
        W1: 重み行列 (d_model × d_ff)
        b1: バイアス (1 × d_ff)
        W2: 重み行列 (d_ff × d_model)
        b2: バイアス (1 × d_model)
    """
    # Step 1: x' × W₁
    hidden = x @ W1           # (7×6) × (6×24) = (7×24)
    # Step 2: + b₁
    hidden = hidden + b1      # (7×24) + (1×24) = (7×24)  ← ブロードキャスト
    # Step 3: ReLU
    hidden = relu(hidden)     # (7×24)
    # Step 4: × W₂
    output = hidden @ W2      # (7×24) × (24×6) = (7×6)
    # Step 5: + b₂
    output = output + b2      # (7×6) + (1×6) = (7×6)  ← ブロードキャスト
    return output

# 全トークンに適用
ffn_output = feed_forward(X_input, W1, b1, W2, b2)

print("=== Feed Forward の全体の結果 ===")
print(f"入力の形状:  {X_input.shape}  (7×6)")
print(f"出力の形状:  {ffn_output.shape}  (7×6)")
print()
print("各ステップの形状変化:")
print(f"  入力 (7×6) → ×W₁ → (7×24) → +b₁ → (7×24) → ReLU → (7×24) → ×W₂ → (7×6) → +b₂ → (7×6)")
print()

# 入力と出力を比較
print("入力と出力の比較:")
print(f"{'トークン':12s} | {'入力':>42s} | {'出力':>42s}")
print("-" * 105)
for i, word in enumerate(words):
    inp = np.round(X_input[i], 3)
    out = np.round(ffn_output[i], 3)
    print(f"{word:12s} | {str(inp):>42s} | {str(out):>42s}")

## 10. Position-wise：トークンごとに独立して適用

Feed Forward は **Position-wise** と呼ばれる通り、
各トークンに **独立に同じ変換** を適用します。

```
"Mount"     → FFN → "Mount" の出力
"Fuji"      → FFN → "Fuji" の出力
"looks"     → FFN → "looks" の出力
"beautiful" → FFN → "beautiful" の出力
"in"        → FFN → "in" の出力
"spring"    → FFN → "spring" の出力
"."         → FFN → "." の出力
```

- 同じ W₁, b₁, W₂, b₂ を全トークンで共有
- トークン間の情報交換はしない（それは Multi-Head Attention の役割）
- 各トークンの特徴を「さらに磨く」処理

In [ ]:
# Position-wise であることを確認
# → 行列演算で全トークンまとめて計算した結果と、1トークンずつ計算した結果が一致する

print("=== Position-wise の確認 ===")
print("全トークンまとめて計算 vs 1トークンずつ計算")
print()

for i, word in enumerate(words):
    # 1トークンずつ計算
    x_single = X_input[i:i+1]  # (1×6)
    out_single = feed_forward(x_single, W1, b1, W2, b2)  # (1×6)
    
    # まとめて計算した結果の i 行目
    out_batch = ffn_output[i:i+1]  # (1×6)
    
    match = np.allclose(out_single, out_batch)
    print(f"  {word:12s}: 一致 = {match}")

print()
print("→ すべて一致！各トークンは独立に計算されている")
print("→ だから Position-wise（位置ごと）と呼ばれる")

In [ ]:
# 次元の拡張と縮小を可視化

fig, axes = plt.subplots(1, 5, figsize=(20, 5),
                         gridspec_kw={'width_ratios': [1, 0.3, 4, 0.3, 1]})

# 入力 (7×6)
ax = axes[0]
im = ax.imshow(X_input, cmap='RdBu_r', aspect='auto')
ax.set_title('入力\n(7×6)', fontsize=12, fontweight='bold')
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=8)
ax.set_xlabel(f'd_model={d_model}', fontsize=10)

# →
ax = axes[1]
ax.text(0.5, 0.5, '→\n×W₁+b₁\nReLU', fontsize=11, ha='center', va='center')
ax.axis('off')

# 中間 (7×24)
ax = axes[2]
hidden_all = relu(X_input @ W1 + b1)  # (7×24)
im = ax.imshow(hidden_all, cmap='RdBu_r', aspect='auto')
ax.set_title('中間層（ReLU後）\n(7×24)', fontsize=12, fontweight='bold')
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=8)
ax.set_xlabel(f'd_ff={d_ff} (= 4 × {d_model})', fontsize=10)

# →
ax = axes[3]
ax.text(0.5, 0.5, '→\n×W₂+b₂', fontsize=11, ha='center', va='center')
ax.axis('off')

# 出力 (7×6)
ax = axes[4]
im = ax.imshow(ffn_output, cmap='RdBu_r', aspect='auto')
ax.set_title('出力\n(7×6)', fontsize=12, fontweight='bold')
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words, fontsize=8)
ax.set_xlabel(f'd_model={d_model}', fontsize=10)

plt.suptitle('Feed Forward: 6次元 → 24次元（拡張）→ 6次元（圧縮）', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("ポイント:")
print(f"  中間層は入力の {d_ff//d_model} 倍の次元（{d_model} → {d_ff}）")
print("  多くのパラメータで処理することで、より豊かな特徴表現を獲得する")

## 11. エンコーダの全体像（図4.48, 4.49）— N回繰り返し

これまでのノートブックで学んできた処理を振り返りましょう。

### エンコーダ1層の処理

```
入力 X (7×6)
  │
  ├──────────────────── Skip Connection ──┐
  ↓                                      │
  Multi-Head Attention (03で学習)          │
  ↓                                      │
  Concat → Linear                        │
  ↓                                      │
  Add ←──────────────────────────────────┘
  ↓
  Layer Norm (05で学習)
  │
  ├──────────────────── Skip Connection ──┐
  ↓                                      │
  Feed Forward (★今回学習)                │
  ↓                                      │
  Add ←──────────────────────────────────┘
  ↓
  Layer Norm
  ↓
  出力 (7×6)
```

### N回繰り返し（図4.49）

上記の処理を **N回繰り返す** のがエンコーダの全体です。
原論文では **N=6** です。

```
入力 (7×6) → [エンコーダ Layer 1] → [Layer 2] → ... → [Layer 6] → 出力 (7×6)
```

出力を入力値にすることで複数回繰り返されます。
**形状が常に 7×6 で変わらない** からこそ、何回でも繰り返すことが可能です。

In [ ]:
# エンコーダ N 層の繰り返しをシミュレーション

def softmax(x):
    """各行に対して Softmax を適用"""
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def layer_norm(x, epsilon=1e-6):
    """Layer Normalization（簡易版: a=1, b=0）"""
    mu = np.mean(x, axis=1, keepdims=True)
    sigma = np.std(x, axis=1, keepdims=True)
    return (x - mu) / (sigma + epsilon)

def multi_head_attention(X, n_heads, d_k, seed_base):
    """Multi-Head Attention（簡易版）"""
    d_model = X.shape[1]
    head_outputs = []
    for h in range(n_heads):
        np.random.seed(seed_base + h * 10)
        W_Q = np.random.randn(d_model, d_k) * 0.3
        W_K = np.random.randn(d_model, d_k) * 0.3
        W_V = np.random.randn(d_model, d_k) * 0.3
        Q = X @ W_Q
        K = X @ W_K
        V = X @ W_V
        attn = softmax(Q @ K.T / np.sqrt(d_k))
        head_outputs.append(attn @ V)
    return np.concatenate(head_outputs, axis=1)

def encoder_layer(X, layer_idx):
    """エンコーダ1層の処理"""
    d_model = X.shape[1]
    d_ff = 4 * d_model
    d_k = d_model // 3
    
    # Multi-Head Attention
    mha_out = multi_head_attention(X, n_heads=3, d_k=d_k, seed_base=layer_idx*100)
    
    # Linear (W_o)
    np.random.seed(layer_idx * 100 + 50)
    W_o = np.random.randn(d_model, d_model) * 0.3
    linear_out = mha_out @ W_o
    
    # Add & Norm (1回目)
    add1 = linear_out + X
    norm1 = layer_norm(add1)
    
    # Feed Forward
    np.random.seed(layer_idx * 100 + 70)
    W1_l = np.random.randn(d_model, d_ff) * 0.3
    b1_l = np.random.randn(1, d_ff) * 0.1
    W2_l = np.random.randn(d_ff, d_model) * 0.3
    b2_l = np.random.randn(1, d_model) * 0.1
    ffn_out = feed_forward(norm1, W1_l, b1_l, W2_l, b2_l)
    
    # Add & Norm (2回目)
    add2 = ffn_out + norm1
    norm2 = layer_norm(add2)
    
    return norm2

# N=6 層のエンコーダを実行
N = 6
print(f"=== エンコーダ {N} 層の繰り返し ===")
print(f"入力の形状: {X_input.shape}  (7×6)")
print()

current = X_input.copy()
layer_outputs = [current.copy()]

for layer in range(N):
    current = encoder_layer(current, layer)
    layer_outputs.append(current.copy())
    print(f"  Layer {layer+1} の出力の形状: {current.shape}  (7×6)  "
          f"平均={np.mean(current):.4f}, 標準偏差={np.std(current):.4f}")

print(f"\n最終出力の形状: {current.shape}  (7×6)")
print("→ 入力と同じ形状！ N回繰り返しても形が変わらない")

In [ ]:
# エンコーダの各層の出力を可視化

fig, axes = plt.subplots(1, N+1, figsize=(22, 5))

for layer in range(N+1):
    ax = axes[layer]
    data = layer_outputs[layer]
    im = ax.imshow(data, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
    
    if layer == 0:
        ax.set_title(f'入力 X', fontsize=11, fontweight='bold')
    else:
        ax.set_title(f'Layer {layer}', fontsize=11, fontweight='bold')
    
    ax.set_yticks(range(n_tokens))
    if layer == 0:
        ax.set_yticklabels(words, fontsize=8)
    else:
        ax.set_yticklabels([])
    ax.set_xticks([])

plt.colorbar(im, ax=axes.tolist(), shrink=0.8)
plt.suptitle(f'図4.49: エンコーダ {N} 層の出力変化（各層 7×6）', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("層を重ねるごとに値のパターンが変化していく")
print("→ 各層が異なる抽象度の特徴を捉えている")

## 12. まとめ

| ポイント | 内容 |
|----------|------|
| **Feed Forward** | エンコーダの後半の処理。各トークンを独立に変換する |
| **式(4-9)** | $FFN(x') = \max(0, x'W_1 + b_1)W_2 + b_2$ |
| **ReLU（式4-10）** | $f(x) = \max(0, x)$ — 負→0、正→そのまま |
| **中間次元 d_ff** | $4 \times d_{model}$ = 4 × 6 = 24（拡張してから戻す）|
| **W₁** | (6×24) — 次元を4倍に拡張 |
| **W₂** | (24×6) — 元の次元に戻す |
| **Position-wise** | 各トークンに独立に同じ変換を適用 |
| **入出力の形状** | 常に 7×6（形状が変わらない）|
| **N回繰り返し** | 原論文では N=6。出力を入力に戻して繰り返す |

### Feed Forward の計算フロー（1トークン分）

```
x' (1×6) → ×W₁ → (1×24) → +b₁ → ReLU → ×W₂ → (1×6) → +b₂ → FFN(x') (1×6)
            拡張              活性化       圧縮
```

### これまでに学んだエンコーダの全体像

| ノートブック | 内容 |
|-------------|------|
| 01 | Transformer の概要 |
| 02 | 単語埋め込み（入力の準備）|
| 03 | Multi-Head Attention（核心部分）|
| 04 | Positional Encoding（位置情報）|
| 05 | Add & Norm（残差接続と正規化）|
| **06** | **Feed Forward（活性化関数による変換）** |

**エンコーダの解説は以上です！**

## 次のステップ

次はいよいよ **デコーダ（Decoder）** に入ります。
自己回帰的なデータ処理によって出力を生成する仕組みを学びます（4-13節）。